# Finetuning the Pre-trained EfficientNet B0 Model to Perform Image Classification for CUB_200_2011

This notebook attempts to fine-tune an EfficientNet B0 model to classify the 200 species of birds from their cropped images. It makes use of the preprocessing done by `notebooks/imagePrep.ipynb`, `scripts/prepare_images.py`, `notebooks/05_image_preparation.ipynb`, `scripts/augmentation.py`. As such, those files must be run in that order beforehand.

Some of the code has been inspired from the tutorial by [Fu (2026)](https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/). This has been marked in the appropriate places.

Table of Contents
1. Understanding our Data
2. 

In [ ]:
# Setting constants to be used later
IMG_SIZE=224 # The width and height of the images
NUM_CLASSES=200 # The number of bird classes

## Section 1: Understanding our Data

Viewing the processed `cub_images_cropped.csv`.

In [15]:
import pandas as pd
img_data = pd.read_csv("../data/processed/cub_images_cropped.csv")

img_data.head()

,image_id,image_path,class_id,species,is_train,bbox_x,bbox_y,bbox_w,bbox_h,bbox_clipped,width,height,mode,cropped_path
0,1,001.Black_footed_Albatross/Black_Footed_Albatr...,1,Black footed Albatross,0,60.0,27.0,325.0,304.0,0,500,335,RGB,images_cropped/001.Black_footed_Albatross/Blac...
1,2,001.Black_footed_Albatross/Black_Footed_Albatr...,1,Black footed Albatross,1,139.0,30.0,153.0,264.0,0,500,336,RGB,images_cropped/001.Black_footed_Albatross/Blac...
2,3,001.Black_footed_Albatross/Black_Footed_Albatr...,1,Black footed Albatross,0,14.0,112.0,388.0,186.0,0,500,347,RGB,images_cropped/001.Black_footed_Albatross/Blac...
3,4,001.Black_footed_Albatross/Black_Footed_Albatr...,1,Black footed Albatross,1,112.0,90.0,255.0,242.0,0,415,500,RGB,images_cropped/001.Black_footed_Albatross/Blac...
4,5,001.Black_footed_Albatross/Black_Footed_Albatr...,1,Black footed Albatross,1,70.0,50.0,134.0,303.0,0,331,380,RGB,images_cropped/001.Black_footed_Albatross/Blac...


# Section 2: Sourcing Image Data

Instead of reworking the image loading logic that has already been coded in other files, this file will borrow their methods. Specifically, the following method adapts the image loading logic used in `notebooks/resnet50_altered_224px.ipynb` written by David Ling Jia Hao.

A separate function is made to convert from the PyTorch style loaders to more Tensorflow loaders. Tensorflow has been chosen for ease of use and familiarity as it was used during the lab sessions.

In [ ]:
# The following code has been inspired by Claude
# It was given other files in this repo to figure out how to leverage them to load images
# Access that conversation here: https://claude.ai/share/fc7c1406-a1ea-49a5-9ec3-177e718d355e
# resnet50_altered_224px.ipynb was heavily referenced too.
# Despite the referencing, all code has been retyped by hand
# and edited in some places where required/desired by myself (Suhaib).

import sys
sys.path.insert(0, "../scripts")

import tensorflow as tf
from augmentation import make_loaders

train_dl, val_dl, test_dl = make_loaders("../data/processed", normalise=False)

# Getting a TF dataset (numpy) using David's loaders from PyTorch
def to_tf(loader):
    def gen():
        for x, y in loader:
            yield x.permute(0, 2, 3, 1).numpy() * 255.0, y.numpy()

    return tf.data.Dataset.from_generator(
        gen,
        output_signature=(tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 3), tf.float32),
                          tf.TensorSpec((None,), tf.int64))
    ).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = to_tf(train_dl), to_tf(val_dl), to_tf(test_dl)

print(train_ds)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


## Section 3: Building the Model

The following function has been inspired by [Fu (2026)](https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/) to build the fine-tuned model's architecture.

In [25]:
from keras.applications import EfficientNetB0
from keras import layers, Model, optimizers

def build_model():
    # Initialise pre-trained model
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    model = EfficientNetB0(include_top=False, input_tensor=inputs, weights="imagenet")

    # Freeze pretrained weights
    model.trainable = False

    x = layers.GlobalAveragePooling2D(name="avg_pool")(model.output)
    x = layers.BatchNormalization()(x)

    top_dropout_rate = 0.2
    x = layers.Dropout(top_dropout_rate, name="top_dropout")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="pred")(x)

    # Compile
    model = Model(inputs, outputs, name="EfficientNet")
    model.compile(
        optimizer = optimizers.Adam(learning_rate=1e-2),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


The following function has been inspired by [Fu (2026)](https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/) to plot historgrams.

In [26]:
# Function to visualise histogram after training
import matplotlib.pyplot as plt

def plot_hist(hist):
    plt.plot(hist.history["accuracy"])
    plt.plot(hist.history["val_accuracy"])
    plt.title("model accuracy")
    plt.ylabel("accuracy")
    plt.xlabel("epoch")
    plt.legend(["train", "validation"], loc="upper left")
    plt.show()

# Section 4: Finetuning

In [ ]:
# Get model
model = build_model()
epochs = 25

hist = model.fit(train_ds, epochs=epochs, validation_data=val_ds)
plot_hist(hist)

Epoch 1/25


c:\Users\suhai\.conda\envs\f21dl-coursework-3.12\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


    149/Unknown 206s 877ms/step - accuracy: 0.3570 - loss: 4.3799

c:\Users\suhai\.conda\envs\f21dl-coursework-3.12\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


149/149 ━━━━━━━━━━━━━━━━━━━━ 290s 1s/step - accuracy: 0.3570 - loss: 4.3799 - val_accuracy: 0.5413 - val_loss: 1.8244
Epoch 2/25
149/149 ━━━━━━━━━━━━━━━━━━━━ 243s 1s/step - accuracy: 0.6424 - loss: 1.7196 - val_accuracy: 0.6155 - val_loss: 1.6600
Epoch 3/25
149/149 ━━━━━━━━━━━━━━━━━━━━ 259s 1s/step - accuracy: 0.7286 - loss: 1.1511 - val_accuracy: 0.6397 - val_loss: 1.6675
Epoch 4/25
149/149 ━━━━━━━━━━━━━━━━━━━━ 262s 1s/step - accuracy: 0.7752 - loss: 0.8677 - val_accuracy: 0.6514 - val_loss: 1.5892
Epoch 5/25
149/149 ━━━━━━━━━━━━━━━━━━━━ 235s 1s/step - accuracy: 0.7978 - loss: 0.7642 - val_accuracy: 0.6564 - val_loss: 1.6789
Epoch 6/25
149/149 ━━━━━━━━━━━━━━━━━━━━ 253s 1s/step - accuracy: 0.7964 - loss: 0.7632 - val_accuracy: 0.6405 - val_loss: 1.7145
Epoch 7/25
149/149 ━━━━━━━━━━━━━━━━━━━━ 241s 1s/step - accuracy: 0.8022 - loss: 0.7557 - val_accuracy: 0.6472 - val_loss: 1.7384
Epoch 8/25
149/149 ━━━━━━━━━━━━━━━━━━━━ 248s 1s/step - accuracy: 0.8240 - loss: 0.6612 - val_accuracy: 0.643

In [ ]:
test_predictions = model.predict(test_ds)